# Evidence Admin Notebook

Manage evidence nodes: **AcademicYear**, **StatusLevel**, **YearSuccessEvidence**.

Evidence consists of documented proof and records that demonstrate progress and compliance with accessibility initiatives.

## Connection Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))

from app.database.graph_schema import set_connection
set_connection()

import pandas as pd
from datetime import date, datetime
pd.set_option('display.max_colwidth', 80)
print('Connection established.')

---
## READ Operations

### List All Academic Years

In [ ]:
from app.database.queries.evidence.read import get_all_academic_years

years = get_all_academic_years()
if years:
    df = pd.DataFrame([{
        'unique_id': y.unique_id,
        'name': y.name,
        'start_date': y.start_date,
        'end_date': y.end_date
    } for y in years])
    display(df)
else:
    print('No AcademicYear nodes found.')

### List All Status Levels

In [ ]:
from app.database.queries.evidence.read import get_all_status_level_nodes

levels = get_all_status_level_nodes()
if levels:
    df = pd.DataFrame([{
        'unique_id': s.unique_id,
        'status_level': s.status_level,
        'status_value': s.status_value,
        'desc_procedures': s.description_of_procedures,
        'desc_documentation': s.description_of_documentation,
        'desc_resources': s.description_of_resources
    } for s in levels])
    display(df)
else:
    print('No StatusLevel nodes found.')

### List All YearSuccessEvidence Nodes

In [ ]:
from app.database.graph_schema import YearSuccessEvidence

yses = YearSuccessEvidence.nodes.all()
if yses:
    df = pd.DataFrame([{
        'unique_id': y.unique_id,
        'year_identifier': y.year_identifier,
        'documentation_status': y.documentation_status,
        'resources_status': y.resources_status,
        'implementation_plan_status': y.implementation_plan_status,
        'admin_review_complete': y.administrative_review_complete,
        'ready_for_admin_review': y.ready_for_admin_review,
        'worked_on_current_year': y.worked_on_in_current_year,
        'will_work_next_year': y.will_work_on_next_year
    } for y in yses])
    display(df)
else:
    print('No YearSuccessEvidence nodes found.')

### Find YSE by Academic Year

In [ ]:
from app.database.queries.evidence.read import find_year_success_evidence_by_academic_year

academic_year = '2024-2025'  # <-- change this

try:
    results = find_year_success_evidence_by_academic_year(academic_year)
    if results:
        display(pd.DataFrame(results))
    else:
        print(f'No YSE nodes for {academic_year}.')
except Exception as e:
    print(f'Error: {e}')

### Get Evidence Trends Between Two Years

In [ ]:
from app.database.queries.evidence.read import get_evidence_trends

past_year = '2023-2024'    # <-- change
current_year = '2024-2025'  # <-- change

try:
    trends = get_evidence_trends(past_year, current_year)
    for wg_name, data in trends.items():
        if wg_name == 'summary':
            print(f'\n=== SUMMARY ===')
            print(f'  Total indicators: {data["total_indicators"]}')
            for wg, stats in data.get('by_working_group', {}).items():
                print(f'  {wg}: {stats}')
        else:
            print(f'\n=== {wg_name} ({len(data)} indicators) ===')
            if data:
                display(pd.DataFrame(data))
except Exception as e:
    print(f'Error: {e}')

### Get Connected Status Levels (with related nodes)

In [ ]:
from app.database.queries.evidence.read import get_connected_status_levels

try:
    connected = get_connected_status_levels()
    for item in connected:
        print(item)
        print('---')
except Exception as e:
    print(f'Error: {e}')

---
## CREATE Operations

### Create YearSuccessEvidence Node

Parameters:
- `academic_year` (str) - e.g. "2024-2025" (must exist as AcademicYear node)
- `success_indicator_composite_key` (str) - e.g. "1.1-web" (must exist as SuccessIndicator)
- `status_level` (str) - e.g. "Not Started", "Initiated", "Defined", "Established", "Managed", "Optimizing"

In [ ]:
from app.database.queries.evidence.create import create_year_success_evidence_node

# create_year_success_evidence_node(
#     academic_year='2024-2025',
#     success_indicator_composite_key='1.1-web',
#     status_level='Not Started'
# )

### Create an Academic Year Node

In [ ]:
from app.database.graph_schema import AcademicYear

# new_year = AcademicYear(
#     name='2025-2026',
#     start_date=date(2025, 8, 1),
#     end_date=date(2026, 7, 31)
# )
# new_year.save()
# print(f'Created AcademicYear: {new_year.name}')

---
## UPDATE Operations

### Assign Implementation to YSE

Parameters:
- `year_success_identifier` (str) - e.g. "2024-2025-1.1-web"
- `implementation_type` (str) - Process, Project, Procedure, Service, Guidance, Tracking, InternalPolicy
- `implementation_title` (str) - title of the implementation node

In [ ]:
from app.database.queries.evidence.update import assign_implementation_to_year_success_indicator

# assign_implementation_to_year_success_indicator(
#     year_success_identifier='2024-2025-1.1-web',
#     implementation_type='Process',
#     implementation_title='Example Process'
# )

### Assign Status Level to YSE

In [ ]:
from app.database.queries.evidence.update import assign_status_to_yse

# assign_status_to_yse(
#     year_success_identifier='2024-2025-1.1-web',
#     status='Established'
# )

### Assign Person / Department / College / Vendor to YSE

In [ ]:
from app.database.queries.evidence.update import (
    assign_person_to_yse,
    assign_department_to_yse,
    assign_college_to_yse,
    assign_vendor_to_yse
)

# assign_person_to_yse('2024-2025-1.1-web', 'John Doe')
# assign_department_to_yse('2024-2025-1.1-web', 'IT Department')
# assign_college_to_yse('2024-2025-1.1-web', 'College of Engineering')
# assign_vendor_to_yse('2024-2025-1.1-web', 'Vendor Name')

### Assign Note / Metric to YSE

In [ ]:
from app.database.queries.evidence.update import assign_note_to_yse, assign_metric_to_yse

# assign_note_to_yse('2024-2025-1.1-web', 'Note Name')
# assign_metric_to_yse('2024-2025-1.1-web', 'metric_composite_key')

### Assign Approver / Update Admin Review

In [ ]:
from app.database.queries.evidence.update import (
    assign_approver_to_yse,
    update_admin_reviewer_description,
    add_admin_reviewer_note
)

# assign_approver_to_yse('2024-2025-1.1-web', employee_id='12345')
# update_admin_reviewer_description('2024-2025-1.1-web', 'Admin review notes here')
# add_admin_reviewer_note('2024-2025-1.1-web', 'Note content', created_by_employee_id='12345')

---
## DELETE Operations

### Unassign Entities from YSE

In [ ]:
from app.database.queries.evidence.delete import (
    unassign_person_from_yse,
    unassign_department_from_yse,
    unassign_vendor_from_yse,
    unassign_college_from_yse,
    unassign_implementation_from_yse
)

# unassign_person_from_yse('2024-2025-1.1-web', 'John Doe')
# unassign_department_from_yse('2024-2025-1.1-web', 'IT Department')
# unassign_vendor_from_yse('2024-2025-1.1-web', 'Vendor Name')
# unassign_college_from_yse('2024-2025-1.1-web', 'College of Engineering')
# unassign_implementation_from_yse('2024-2025-1.1-web', 'Process', 'Example Process')

### Delete YearSuccessEvidence Node

In [ ]:
from app.database.queries.evidence.delete import delete_year_success_evidence

# delete_year_success_evidence('2024-2025-1.1-web')

---
## RELATIONSHIP Management

### View All Relationships for a YSE Node

In [ ]:
from app.database.graph_schema import YearSuccessEvidence

yse_id = 'PASTE_YEAR_IDENTIFIER_HERE'  # <-- change

try:
    yse = YearSuccessEvidence.nodes.get(year_identifier=yse_id)
    print(f'=== {yse.year_identifier} ===')
    print(f'\nAcademic Year: {[y.name for y in yse.academic_year.all()]}')
    print(f'Status Level: {[s.status_level for s in yse.status_level.all()]}')
    print(f'Tracks SI: {[si.composite_key for si in yse.tracks_success_indicator.all()]}')
    print(f'\nNotes ({len(yse.notes.all())}): {[n.name for n in yse.notes.all()]}')
    print(f'Messages ({len(yse.messages.all())}): {[m.name for m in yse.messages.all()]}')
    print(f'Metrics ({len(yse.metrics.all())}): {[m.name for m in yse.metrics.all()]}')
    print(f'\nProcesses: {[p.title for p in yse.processes_that_evidence.all()]}')
    print(f'Projects: {[p.title for p in yse.projects_that_evidence.all()]}')
    print(f'Procedures: {[p.title for p in yse.procedures_that_evidence.all()]}')
    print(f'Services: {[s.title for s in yse.services_that_evidence.all()]}')
    print(f'Guidance: {[g.title for g in yse.guidance_that_evidence.all()]}')
    print(f'Trackings: {[t.title for t in yse.trackings_that_evidence.all()]}')
    print(f'Internal Policies: {[p.title for p in yse.internal_policies_that_evidence.all()]}')
    print(f'\nPersons implementing: {[p.name for p in yse.persons_that_implement.all()]}')
    print(f'Admin reviewers: {[p.name for p in yse.assigned_reviewers.all()]}')
except Exception as e:
    print(f'Error: {e}')